In [20]:
import pandas as pd

df_raw = pd.read_pickle("processed_data/dataset_raw_datpacket_user.pkl")
print(len(df_raw))

4045738


In [21]:
df_raw.keys()

Index(['Object', 'Time Step', 'Id', 'User', 'Application', 'Size', 'Status',
       'Queue Delay', 'Transmission Delay', 'Processing Delay',
       'Propagation Delay', 'Total Delay', 'Total Path', 'Hops', 'Object_user',
       'Time Step_user', 'Instance ID', 'Coordinates', 'Base Station',
       'Delays', 'Delay SLAs', 'Communication Paths', 'Making Requests',
       'Access History', 'Distribution', 'Scenario', 'App_ms', 'Run'],
      dtype='str')

In [22]:
columns_to_keep = [

    'Distribution', 'Scenario', 'App_ms', 'Run', 'Time Step', 'Object', 'Id',

    # DataPacket
    'Status',
    'User',
    'Application',
    'Processing Delay',
    'Propagation Delay',
    'Total Delay',
    'Total Path',
    'Hops',

    # User
    'Delays', 'Delay SLAs',

]

df_light = df_raw[columns_to_keep].copy()

In [23]:
user_data_SLA = df_light.groupby(['Scenario', 'App_ms'])['Delay SLAs'].first().apply(lambda d: list(d.values())[0]).reset_index()
user_data_SLA.head()

,Scenario,App_ms,Delay SLAs
0,1_26_solution_v0,1SMM,63.486
1,1_26_solution_v0,1SMS,63.486
2,1_26_solution_v0,1SSM,63.486
3,1_26_solution_v0,1SSS,63.486
4,1_26_solution_v10,1MMM,69.112


In [49]:
df_packets = df_light[df_light["Status"] == "finished"].copy()

if 'Delay SLAs' in df_packets.columns:
    df_packets = df_packets.drop(columns=['Delay SLAs'])

df_packets_sla = df_packets.merge(user_data_SLA, on=['Scenario', 'App_ms'], how='left')

columns_to_keep = [

    'Distribution', 'Scenario', 'App_ms', 'Run', 'Time Step', 'Object', 'Id',

    # DataPacket
    'Status',
    'User',
    'Application',
    'Processing Delay',
    'Propagation Delay',
    'Total Delay',
    'Delay SLAs', # aggiunto da user 
    'Total Path',
    'Hops',
]

df_packets_sla = df_packets_sla[columns_to_keep].copy()
# df_packets_sla[(df_packets_sla['Scenario'] == '2_26_solution_v2') & 
#                (df_packets_sla['Run'] == 'run_7') & 
#                (df_packets_sla['App_ms'] == '2SSS') & 
#                (df_packets_sla['Distribution'] == 'exponential') &
#                (df_packets_sla['Application'] == 3)]
df_packets_sla

,Distribution,Scenario,App_ms,Run,Time Step,Object,Id,Status,User,Application,Processing Delay,Propagation Delay,Total Delay,Delay SLAs,Total Path,Hops
0,uniform,1_26_solution_v3,1MMM,run_7,36,DataPacket_1,1,finished,1,1,24,28,52,60.273,"[[11], [11], [11, 23], [23, 7, 24, 21], [21, 2...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
1,uniform,1_26_solution_v3,1MMM,run_7,36,DataPacket_1,1,finished,1,1,24,28,52,60.273,"[[11], [11], [11, 23], [23, 7, 24, 21], [21, 2...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
2,uniform,1_26_solution_v3,1MMM,run_7,36,DataPacket_1,1,finished,1,1,24,28,52,60.273,"[[11], [11], [11, 23], [23, 7, 24, 21], [21, 2...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
3,uniform,1_26_solution_v3,1MMM,run_7,36,DataPacket_1,1,finished,1,1,24,28,52,60.273,"[[11], [11], [11, 23], [23, 7, 24, 21], [21, 2...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
4,uniform,1_26_solution_v3,1MMM,run_7,36,DataPacket_1,1,finished,1,1,24,28,52,60.273,"[[11], [11], [11, 23], [23, 7, 24, 21], [21, 2...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335156,exponential,1_26_solution_v0,1SMM,run_6,42,DataPacket_1,1,finished,1,1,31,27,58,63.486,"[[14], [14], [14, 0], [0, 16], [16, 17], [17, ...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
335157,exponential,1_26_solution_v0,1SMM,run_6,42,DataPacket_1,1,finished,1,1,31,27,58,63.486,"[[14], [14], [14, 0], [0, 16], [16, 17], [17, ...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
335158,exponential,1_26_solution_v0,1SMM,run_6,42,DataPacket_1,1,finished,1,1,31,27,58,63.486,"[[14], [14], [14, 0], [0, 16], [16, 17], [17, ...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
335159,exponential,1_26_solution_v0,1SMM,run_6,42,DataPacket_1,1,finished,1,1,31,27,58,63.486,"[[14], [14], [14, 0], [0, 16], [16, 17], [17, ...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."


In [51]:
idx_max = df_packets_sla.groupby(['Distribution', 'Scenario', 'App_ms', 'Run'])['Total Delay'].idxmax()

df_worst_case = df_packets_sla.loc[idx_max].reset_index(drop=True).copy()

# df_worst_case[(df_worst_case['Scenario'] == '2_26_solution_v2') & 
#                (df_worst_case['Run'] == 'run_7') & 
#                (df_worst_case['App_ms'] == '2SSS') & 
#                (df_worst_case['Distribution'] == 'exponential')]
df_worst_case

,Distribution,Scenario,App_ms,Run,Time Step,Object,Id,Status,User,Application,Processing Delay,Propagation Delay,Total Delay,Delay SLAs,Total Path,Hops
0,exponential,1_26_solution_v0,1SMM,run_0,56,DataPacket_1,1,finished,1,1,45,27,72,63.486,"[[14], [14], [14, 0], [0, 16], [16, 17], [17, ...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
1,exponential,1_26_solution_v0,1SMM,run_1,21,DataPacket_1,1,finished,1,1,10,27,37,63.486,"[[14], [14], [14, 0], [0, 16], [16, 17], [17, ...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
2,exponential,1_26_solution_v0,1SMM,run_2,33,DataPacket_1,1,finished,1,1,22,27,49,63.486,"[[14], [14], [14, 0], [0, 16], [16, 17], [17, ...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
3,exponential,1_26_solution_v0,1SMM,run_3,36,DataPacket_1,1,finished,1,1,25,27,52,63.486,"[[14], [14], [14, 0], [0, 16], [16, 17], [17, ...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
4,exponential,1_26_solution_v0,1SMM,run_4,32,DataPacket_1,1,finished,1,1,21,27,48,63.486,"[[14], [14], [14, 0], [0, 16], [16, 17], [17, ...","[{'hop_index': 0, 'link_index': 0, 'source': 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,uniform,2_26_solution_v7,2SMS,run_5,37,DataPacket_2,2,finished,2,2,24,36,60,70.427,"[[24], [24], [24, 21], [21, 16], [16, 12], [12...","[{'hop_index': 0, 'link_index': 0, 'source': 2..."
1436,uniform,2_26_solution_v7,2SMS,run_6,43,DataPacket_2,2,finished,2,2,30,36,66,70.427,"[[24], [24], [24, 21], [21, 16], [16, 12], [12...","[{'hop_index': 0, 'link_index': 0, 'source': 2..."
1437,uniform,2_26_solution_v7,2SMS,run_7,44,DataPacket_1,1,finished,1,1,29,35,64,70.427,"[[22], [22, 16, 21], [21, 16, 5], [5, 16], [16...","[{'hop_index': 0, 'link_index': 0, 'source': 2..."
1438,uniform,2_26_solution_v7,2SMS,run_8,47,DataPacket_2,2,finished,2,2,34,36,70,70.427,"[[24], [24], [24, 21], [21, 16], [16, 12], [12...","[{'hop_index': 0, 'link_index': 0, 'source': 2..."


In [54]:
df_worst_case['SLA_Violation'] = df_worst_case['Total Delay'] > df_worst_case['Delay SLAs']

df_worst_case['SLA_Violation_Percentage'] = ((df_worst_case['Total Delay'] - df_worst_case['Delay SLAs']) / df_worst_case['Delay SLAs'] * 100).round(2)

cols_final = [
    'Distribution', 'Scenario', 'App_ms', 'Run', 
    'Total Delay', 'Delay SLAs', 'SLA_Violation', 'SLA_Violation_Percentage',
    'Processing Delay', 'Propagation Delay', 'Hops'
]

df_worst_case[cols_final].to_pickle("processed_data/dataset_worst_case_datapacket_user.pkl")

df_worst_case[cols_final]

,Distribution,Scenario,App_ms,Run,Total Delay,Delay SLAs,SLA_Violation,SLA_Violation_Percentage,Processing Delay,Propagation Delay,Hops
0,exponential,1_26_solution_v0,1SMM,run_0,72,63.486,True,13.41,45,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
1,exponential,1_26_solution_v0,1SMM,run_1,37,63.486,False,-41.72,10,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
2,exponential,1_26_solution_v0,1SMM,run_2,49,63.486,False,-22.82,22,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
3,exponential,1_26_solution_v0,1SMM,run_3,52,63.486,False,-18.09,25,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
4,exponential,1_26_solution_v0,1SMM,run_4,48,63.486,False,-24.39,21,27,"[{'hop_index': 0, 'link_index': 0, 'source': 1..."
...,...,...,...,...,...,...,...,...,...,...,...
1435,uniform,2_26_solution_v7,2SMS,run_5,60,70.427,False,-14.81,24,36,"[{'hop_index': 0, 'link_index': 0, 'source': 2..."
1436,uniform,2_26_solution_v7,2SMS,run_6,66,70.427,False,-6.29,30,36,"[{'hop_index': 0, 'link_index': 0, 'source': 2..."
1437,uniform,2_26_solution_v7,2SMS,run_7,64,70.427,False,-9.13,29,35,"[{'hop_index': 0, 'link_index': 0, 'source': 2..."
1438,uniform,2_26_solution_v7,2SMS,run_8,70,70.427,False,-0.61,34,36,"[{'hop_index': 0, 'link_index': 0, 'source': 2..."
